In [18]:
import sys
sys.path.append("src")
from loan_default_ml_pipeline.ingestion.load_data import load_raw_data
from loan_default_ml_pipeline.validation.data_quality import null_summary

df = load_raw_data()
# apply filter of default binary loan_statuses
df_filt = df[(df['loan_status'].isin(['Charged Off', 'Late (31-120 days)', 'Late (16-30 days)', 'Fully Paid']))]
null_summary(df_filt)


,missing_count,percent_missing,dtype
verification_income_joint,481,86.200717,object
debt_to_income_joint,479,85.842294,float64
annual_income_joint,479,85.842294,float64
months_since_90d_late,437,78.315412,float64
months_since_last_delinq,310,55.555556,float64
months_since_last_credit_inquiry,52,9.318996,float64
emp_title,38,6.810036,object
emp_length,37,6.630824,float64
num_accounts_120d_past_due,20,3.584229,float64
debt_to_income,2,0.358423,float64


In [ ]:
# Check if the large null counts from *_joint columns coincide with application_type = 'joint'
import pandas as pd
results = []

for col in [c for c in df.columns if c.endswith('_joint')]:
    results.append({
        "column": col,
        "missing_when_joint": df[(df[col].isnull()) & \
                                 (df['application_type'] == 'joint')].shape[0],
        "present_when_individual": df[(df[col].notnull()) & \
                                      (df['application_type'] == 'individual')].shape[0]
    })

pd.DataFrame(results)

,column,missing_when_joint,present_when_individual
0,annual_income_joint,0,0
1,verification_income_joint,40,0
2,debt_to_income_joint,0,0


In [19]:
df_filt.describe()

,emp_length,annual_income,debt_to_income,annual_income_joint,debt_to_income_joint,delinq_2y,months_since_last_delinq,earliest_credit_line,inquiries_last_12m,total_credit_lines,...,public_record_bankrupt,loan_amount,term,interest_rate,installment,balance,paid_total,paid_principal,paid_interest,paid_late_fees
count,521.000000,5.580000e+02,556.000000,79.000000,79.000000,558.000000,248.000000,558.000000,558.000000,558.000000,...,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000
mean,5.844530,8.492513e+04,17.355378,136008.340127,18.525443,0.222222,33.487903,2001.313620,2.279570,23.510753,...,0.127240,16015.277778,43.870968,14.001183,471.738477,3266.548835,12967.658142,12595.370090,371.680609,0.607258
std,3.800407,1.090174e+05,12.941129,72046.713089,8.894771,0.556413,21.405282,7.636739,2.455488,12.590458,...,0.338881,10672.550458,11.277367,5.716751,307.972645,8344.893022,11322.193360,11163.715109,458.395462,4.002370
min,0.000000,0.000000e+00,0.000000,26400.000000,1.350000,0.000000,1.000000,1965.000000,0.000000,2.000000,...,0.000000,1000.000000,36.000000,5.310000,33.980000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,4.800000e+04,9.550000,93698.980000,12.170000,0.000000,16.000000,1998.000000,1.000000,14.000000,...,0.000000,7500.000000,36.000000,9.920000,242.230000,0.000000,3223.099316,3000.000000,47.565000,0.000000
50%,6.000000,6.885000e+04,16.160000,116000.000000,19.050000,0.000000,30.000000,2003.000000,2.000000,22.000000,...,0.000000,13000.000000,36.000000,13.590000,379.135000,0.000000,10226.098209,10000.000000,199.610000,0.000000
75%,10.000000,9.787500e+04,23.487500,161100.000000,23.895000,0.000000,45.000000,2006.000000,3.000000,30.000000,...,0.000000,24000.000000,60.000000,17.090000,673.405000,0.000000,20114.215801,20000.000000,498.047500,0.000000
max,10.000000,2.300000e+06,209.100000,395000.000000,39.790000,4.000000,109.000000,2014.000000,20.000000,80.000000,...,2.000000,40000.000000,60.000000,30.940000,1503.890000,40000.000000,41630.443684,40000.000000,3080.080000,45.120000


In [9]:
df.columns.tolist()

['emp_title',
 'emp_length',
 'state',
 'homeownership',
 'annual_income',
 'verified_income',
 'debt_to_income',
 'annual_income_joint',
 'verification_income_joint',
 'debt_to_income_joint',
 'delinq_2y',
 'months_since_last_delinq',
 'earliest_credit_line',
 'inquiries_last_12m',
 'total_credit_lines',
 'open_credit_lines',
 'total_credit_limit',
 'total_credit_utilized',
 'num_collections_last_12m',
 'num_historical_failed_to_pay',
 'months_since_90d_late',
 'current_accounts_delinq',
 'total_collection_amount_ever',
 'current_installment_accounts',
 'accounts_opened_24m',
 'months_since_last_credit_inquiry',
 'num_satisfactory_accounts',
 'num_accounts_120d_past_due',
 'num_accounts_30d_past_due',
 'num_active_debit_accounts',
 'total_debit_limit',
 'num_total_cc_accounts',
 'num_open_cc_accounts',
 'num_cc_carrying_balance',
 'num_mort_accounts',
 'account_never_delinq_percent',
 'tax_liens',
 'public_record_bankrupt',
 'loan_purpose',
 'application_type',
 'loan_amount',
 'term'

In [12]:
df['loan_status'].value_counts()

loan_status
Current               9375
Fully Paid             447
In Grace Period         67
Late (31-120 days)      66
Late (16-30 days)       38
Charged Off              7
Name: count, dtype: int64

In [14]:
# Filter to only default binary statuses
cohort = df[df['loan_status'].isin(['Charged Off',
                                    'Late (31-120 days)',
                                    'Late (16-30 days)',
                                    'Fully Paid'])]

# Analyze chosen signals across each loan_status
cohort.groupby('loan_status').agg(
    record_count = ('loan_status', 'count'),
    joint_app_count = ('application_type', lambda x: (x == 'joint').sum()),
    avg_30d_past_due = ('num_accounts_30d_past_due', 'mean'),
    avg_120d_past_due = ('num_accounts_120d_past_due', 'mean'),
    avg_months_deliq = ('months_since_last_delinq', 'mean'),
    pct_ever_deliq = ('months_since_last_delinq', lambda x: (x.notna().sum() / len(x) * 100).round(2))
    ).round(3)

,record_count,joint_app_count,avg_30d_past_due,avg_120d_past_due,avg_months_deliq,pct_ever_deliq
loan_status,,,,,,
Charged Off,7,1,0.0,0.0,33.500,28.57
Fully Paid,447,59,0.0,0.0,33.733,45.19
Late (16-30 days),38,7,0.0,0.0,31.158,50.00
Late (31-120 days),66,12,0.0,0.0,33.280,37.88
